# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema standard.

### Dataset Source
The dataset schema is provided via the following Croissant JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # To suppress warnings for cleaner output

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata and object
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

# Print out dataset name and description
print(f"{meta.name}: {meta.description}")
print(f"\nCite as: {meta.citeAs}")
print(f"Published: {meta.datePublished}. License: {meta.license}")

## 2. Data Overview
Discover available record sets, fields, fields' types, and their IDs. All elements are referenced by their Croissant `@id` identifiers.

In [ ]:
# Discover all record sets in the dataset metadata (usually in Dataset.recordSet)
record_sets = getattr(meta, 'recordSet', None) or []

if not isinstance(record_sets, list):
    record_sets = [record_sets]

# Display record set IDs
if not record_sets or len(record_sets) == 0:
    print("No record sets defined in metadata; attempting to find via dataset introspection...")
    # Try to find available record sets from the dataset object
    available_record_sets = list(dataset.record_sets.keys())
    print("Found record set @ids in dataset:")
    for rec_id in available_record_sets:
        print(f"  - {rec_id}")
    record_sets = available_record_sets
else:
    print("Record sets defined in Croissant metadata:")
    for rs in record_sets:
        print(f"  - {getattr(rs, '@id', rs)}")

# For each record set, print its fields
for rs_id in record_sets:
    print(f"\nFields in RecordSet '@id': {rs_id}")
    try:
        # mlcroissant provides Dataset.fields(record_set=rs_id) which yields field dicts
        fields = list(dataset.fields(record_set=rs_id))
        for field in fields:
            name = field.get('name')
            field_id = field.get('@id')
            dtype = field.get('dataType','')
            print(f"    {name}  (@id: {field_id})  type: {dtype}")
    except Exception as e:
        print("    Could not fetch fields (", str(e), ")")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Tip:** All further references to record sets and fields use their exact `@id` values.

In [ ]:
# Extract all available record sets into pandas DataFrames by @id
dataframes = {}

for rec_set_id in record_sets:
    print(f"Loading records for record set: {rec_set_id}")
    try:
        recs = list(dataset.records(record_set=rec_set_id))
        if len(recs) > 0:
            df = pd.DataFrame(recs)
            dataframes[rec_set_id] = df
            print(f"  Loaded {len(df)} records. Columns: {list(df.columns)}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error: {e}")

# Pick the primary tabular record set (typically only one in a biomedical tabular dataset)
if len(dataframes) > 0:
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"\nPrimary record set selected for analysis: {primary_record_set_id}")
    print(f"Columns: {list(dataframes[primary_record_set_id].columns)}")
    display(dataframes[primary_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Here we demonstrate common data wrangling with Croissant field IDs. Filtering, normalization, and group-by aggregation use field references by their Croissant `@id`.

We'll select a numeric field and a group/categorical field for demonstration. Update `numeric_field_id` and `group_field_id` below as needed for your exploration.

In [ ]:
# Use the primary record set for EDA
df = dataframes[primary_record_set_id]
print(f"Fields available in record set '{primary_record_set_id}':\n[df.columns.tolist()]")

# Select field IDs (replace as appropriate):
# Example guesses for field IDs (you MUST replace with the correct @id from your dataset!):
numeric_field_id = None
group_field_id = None

# Heuristically try finding a numeric field by field type
fields_metadata = list(dataset.fields(record_set=primary_record_set_id))
for f in fields_metadata:
    dtype = f.get('dataType','')
    field_id = f.get('@id')
    if dtype in ('Integer','Float','Number') and field_id in df.columns:
        numeric_field_id = field_id
        print(f"Auto-selected numeric field: {numeric_field_id}")
        break

# Find a suitable groupby/categorical field
for f in fields_metadata:
    dtype = f.get('dataType','')
    field_id = f.get('@id')
    if dtype in ('Text', 'String') and field_id in df.columns:
        group_field_id = field_id
        print(f"Auto-selected group field: {group_field_id}")
        break

# If none were auto-selected fallback to user input (raise error)
if numeric_field_id is None or group_field_id is None:
    raise ValueError('Could not auto-select numeric or group fields by @id. Please update numeric_field_id and group_field_id to match your column @ids.')

# Filter: example threshold (median for demo)
if numeric_field_id in df:
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (median):\n{filtered_df.shape[0]} records.")
    
    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())
else:
    print(f"Numeric field {numeric_field_id} not in DataFrame.")

# Group by the group_field_id (if exists)
if group_field_id in filtered_df:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df.head())
else:
    print(f"Group field {group_field_id} not in filtered DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields. We use the Croissant field `@id`s for all references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Grouped barplot if grouping field is categorical with not too many levels
if group_field_id in df and df[group_field_id].nunique() < 20:
    plt.figure(figsize=(10,5))
    sns.barplot(
        data=grouped_df,
        x=group_field_id, y=numeric_field_id
    )
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded, explored, and briefly analyzed the FAIR^2 CRC Survivor dataset using the `mlcroissant` library by referencing all entities and fields using their Croissant `@id`. You can use this template to perform deeper analysis or adapt it to other Croissant-compatible datasets.

### Key points:
- All table and field references use the Croissant schema `@id` for repeatable, schema-robust access.
- You can filter, normalize, and group fields dynamically by updating the appropriate `@id` variables.
- Visualizations can be made for any quantitative or grouping field using the same approach.

For more advanced uses and documentation, see the [`mlcroissant` documentation](https://github.com/mlcommons/croissant) and the [Croissant specification](https://mlcommons.org/croissant/spec/).